# Poker Hand Evaluation

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import json

from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
RANDOM_STATE = 1337

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "padme" / "src" / "main" / "resources" / "data"
OUT_DIR = DATA_DIR / "output/poker"

TRAIN_PATH = DATA_DIR / "input" / "poker_train.csv"
TEST_PATH = OUT_DIR / "poker_test.csv"

BASELINE_MODE = "baseline"
MODES = ["k_center", "graph_cut"]
SWEEP_MODES = [m for m in MODES if m != BASELINE_MODE]

MODE_LABELS = {
    "baseline": "Baseline",
    "random": "Random",
    "padme": "PADME",
    "graph_cut": "Graph Cut",
    "k_center": "K-Center",
}

MODE_STYLE = {
    "random": {"color": "tab:orange", "marker": "s"},
    "padme": {"color": "gold", "marker": "o"},
    "graph_cut": {"color": "tab:red", "marker": "^"},
    "k_center": {"color": "tab:blue", "marker": "v"},
}

NODES = 5
BASELINE_NODE = 0

RATIOS = [0.01, 0.02, 0.03, 0.04, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60]

DROP_COLS = ["__id", "label"]
LABEL_COL = "label"

rcParams["figure.facecolor"] = "white"
rcParams["axes.facecolor"] = "white"
rcParams["savefig.facecolor"] = "white"
rcParams["text.color"] = "black"
rcParams["axes.labelcolor"] = "black"
rcParams["axes.edgecolor"] = "#444444"
rcParams["xtick.color"] = "#444444"
rcParams["ytick.color"] = "#444444"
rcParams["legend.facecolor"] = "white"
rcParams["legend.edgecolor"] = "#cccccc"

In [ ]:
def ratio_to_int(r: float) -> int:
    return int(round(r * 100))

def load_csv_any(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Not found: {path}")
    return pd.read_csv(path)

def load_test(train_csv: Path, test_csv: Path):
    if not test_csv.exists():
        raise FileNotFoundError(f"Required test set not found: {test_csv}. ")

    df_train = load_csv_any(train_csv)
    df_test = load_csv_any(test_csv)

    y_train = df_train[LABEL_COL].astype(int).to_numpy()
    X_train = df_train.drop(columns=[c for c in [LABEL_COL] + DROP_COLS if c in df_train.columns])

    y_test = df_test[LABEL_COL].astype(int).to_numpy()
    X_test = df_test.drop(columns=[c for c in [LABEL_COL] + DROP_COLS if c in df_test.columns])

    return X_train, y_train, X_test, y_test

X_train_full, y_train_full, X_test, y_test = load_test(TRAIN_PATH, TEST_PATH)

feature_cols = list(X_train_full.columns)

print("Train full:", X_train_full.shape, "Test:", X_test.shape, "Classes:", sorted(pd.Series(y_test).unique().tolist()))


def discover_mode_ratios(output_root: Path, mode: str):
    mode_dir = Path(output_root) / mode
    ratios = []
    if not mode_dir.exists():
        return ratios

    for child in mode_dir.iterdir():
        if not child.is_dir():
            continue
        try:
            ratios.append(int(child.name) / 100.0)
        except ValueError:
            continue

    return sorted(set(ratios))


def discover_available_ratios(output_root: Path, modes=None):
    modes = SWEEP_MODES if modes is None else modes
    return {mode: discover_mode_ratios(output_root, mode) for mode in modes}


AVAILABLE_RATIOS_BY_MODE = discover_available_ratios(OUT_DIR)
COMMON_SWEEP_RATIOS = sorted(
    set.intersection(*(set(ratios) for ratios in AVAILABLE_RATIOS_BY_MODE.values()))
) if AVAILABLE_RATIOS_BY_MODE and all(AVAILABLE_RATIOS_BY_MODE.values()) else []

print("Available ratios by mode:", AVAILABLE_RATIOS_BY_MODE)
print("Common ratios across sweep modes:", COMMON_SWEEP_RATIOS)


In [ ]:
HIST_RATIO = 0.20
SAVE_DPI = 200

def load_metrics_json(path):
    p = Path(path)
    if not p.exists():
        return None
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def load_mode_series(output_root: Path, mode: str):
    mode_dir = Path(output_root) / mode
    if not mode_dir.exists():
        return [], []

    pairs = []
    for child in mode_dir.iterdir():
        if not child.is_dir():
            continue
        metrics = load_metrics_json(child / "metrics.json")
        if metrics is None:
            continue
        pairs.append((float(metrics["keepRatio"]), int(metrics["totalBytesSent"])))

    pairs.sort(key=lambda x: x[0])
    ratios = [x[0] for x in pairs]
    bytes_sent = [x[1] for x in pairs]
    return ratios, bytes_sent

def load_network_bytes_data(output_root: Path, modes=None):
    output_root = Path(output_root)
    modes = SWEEP_MODES if modes is None else modes

    baseline_metrics = load_metrics_json(output_root / BASELINE_MODE / "metrics.json")
    baseline_bytes = None if baseline_metrics is None else baseline_metrics["totalBytesSent"]

    mode_data = {}
    for mode in modes:
        ratios, bytes_sent = load_mode_series(output_root, mode)
        mode_data[mode] = {"ratios": ratios, "bytes": bytes_sent}

    return baseline_bytes, mode_data

baseline_bytes, bytes_data = load_network_bytes_data(OUT_DIR)

# prints
print("\n=== Total bytes sent ===")
if baseline_bytes is not None:
    print(f"{MODE_LABELS[BASELINE_MODE]}: {baseline_bytes}")
else:
    print(f"{MODE_LABELS[BASELINE_MODE]}: not found")

for mode in SWEEP_MODES:
    ratios = bytes_data[mode]["ratios"]
    bytes_sent = bytes_data[mode]["bytes"]

    print(f"\n{MODE_LABELS[mode]}:")
    if not ratios:
        print("  no data")
        continue

    for r, b in zip(ratios, bytes_sent):
        print(f"  keep_ratio={r:.4f}  totalBytesSent={b}")

fig = plt.figure(figsize=(7, 6))

x_for_line = sorted({r for mode_data in bytes_data.values() for r in mode_data["ratios"]})
if baseline_bytes is not None:
    if not x_for_line:
        x_for_line = [0.0, 1.0]
    plt.plot(
        x_for_line,
        [baseline_bytes] * len(x_for_line),
        linestyle="--",
        linewidth=2,
        color="black",
        label=MODE_LABELS[BASELINE_MODE]
    )

for mode in SWEEP_MODES:
    ratios = bytes_data[mode]["ratios"]
    bytes_sent = bytes_data[mode]["bytes"]
    if not ratios:
        continue

    plt.plot(
        ratios,
        bytes_sent,
        marker=MODE_STYLE[mode]["marker"],
        linewidth=2,
        color=MODE_STYLE[mode]["color"],
        label=MODE_LABELS[mode]
    )

plt.xlabel("Keep ratio")
plt.ylabel("Total bytes sent")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def load_time_multipliers(output_root: Path, modes=None, reference_mode="random"):
    output_root = Path(output_root)
    modes = SWEEP_MODES if modes is None else modes

    def load_mode_times(mode: str):
        mode_dir = output_root / mode
        if not mode_dir.exists():
            return {}

        times = {}
        for child in mode_dir.iterdir():
            if not child.is_dir():
                continue
            try:
                ratio = int(child.name) / 100.0
            except ValueError:
                continue
            metrics = load_metrics_json(child / "metrics.json")
            if metrics is None or "simulationTimeSeconds" not in metrics:
                continue
            times[ratio] = float(metrics["simulationTimeSeconds"])
        return times

    mode_times = {mode: load_mode_times(mode) for mode in modes}
    common_ratios = sorted(
        set.intersection(*(set(times.keys()) for times in mode_times.values()))
    ) if mode_times and all(mode_times.values()) else []

    if reference_mode not in mode_times or not common_ratios:
        return None

    ref_total_time = float(np.sum([mode_times[reference_mode][r] for r in common_ratios]))
    if ref_total_time == 0:
        return None

    labels = [MODE_LABELS[reference_mode]]
    multipliers = [1.0]
    colors = [MODE_STYLE[reference_mode]["color"]]

    for mode in modes:
        if mode == reference_mode:
            continue

        mode_total_time = float(np.sum([mode_times[mode][r] for r in common_ratios]))
        labels.append(MODE_LABELS[mode])
        multipliers.append(mode_total_time / ref_total_time)
        colors.append(MODE_STYLE[mode]["color"])

    return labels, multipliers, colors, common_ratios


time_data = load_time_multipliers(OUT_DIR, modes=SWEEP_MODES, reference_mode="random")

if time_data is not None:
    labels, multipliers, colors, common_ratios = time_data

    fig = plt.figure(figsize=(7, 6))
    bars = plt.bar(labels, multipliers, edgecolor="black", color=colors)

    ymax = max(multipliers) if multipliers else 1.0

    for bar, mult in zip(bars, multipliers):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005 * ymax,
            f"{mult:.1f}x",
            ha="center",
            va="bottom",
            fontsize=11
        )

    plt.ylabel("Total simulation time multiplier vs random")
    plt.tight_layout()
    plt.show()

    print("Time plot ratios (common to all sweep modes):", common_ratios)


In [ ]:
def node_file(mode: str, ratio_int: int | None, node_idx: int) -> Path:
    if mode == BASELINE_MODE:
        return OUT_DIR / BASELINE_MODE / f"{BASELINE_MODE}_node{node_idx}.csv"
    return OUT_DIR / mode / str(ratio_int) / f"{mode}_node{node_idx}.csv"

def load_node_dataset(mode: str, ratio: float | None, node_idx: int):
    r_int = None if ratio is None else ratio_to_int(ratio)
    p = node_file(mode, r_int, node_idx)

    df = load_csv_any(p)
    if LABEL_COL not in df.columns:
        raise ValueError(f"{p} does not have column '{LABEL_COL}'.")

    y = df[LABEL_COL].astype(int).to_numpy()
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    X = X.reindex(columns=feature_cols, fill_value=0.0)

    return X, y, str(p)

In [ ]:
stats_rows = []

for mode in MODES:
    if mode == BASELINE_MODE:
        ratios_to_check = [None]
    else:
        ratios_to_check = [0.01, 0.02, 0.03, 0.04, 0.05, 0.08, 0.10]

    for r in ratios_to_check:
        for i in range(NODES):
            Xn, yn, pn = load_node_dataset(mode, r, i)

            yn_series = pd.Series(yn)
            class_counts = yn_series.value_counts().sort_index().to_dict()
            total = int(len(yn_series))

            stats_rows.append({
                "mode": mode,
                "keep_ratio": 1.0 if r is None else float(r),
                "ratio_int": 100 if r is None else ratio_to_int(r),
                "node": i,
                "rows": total,
                "n_classes": int(yn_series.nunique()),
                "class_counts": class_counts,
                "file": str(pn)
            })

stats_df = pd.DataFrame(stats_rows).sort_values(
    ["mode", "keep_ratio", "node"]
).reset_index(drop=True)

stats_df

In [ ]:
plt.rcdefaults()
plt.style.use("default")

baseline_path = OUT_DIR / BASELINE_MODE / f"{BASELINE_MODE}_node0.csv"


def _load_geom_matrix(path):
    df = load_csv_any(path)
    X = df.drop(columns=[c for c in [LABEL_COL] + DROP_COLS if c in df.columns], errors="ignore")
    X = X.reindex(columns=feature_cols, fill_value=0.0)
    X = X.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df, X


def _fit_pca_reference(X_base):
    Xb = X_base.to_numpy(dtype=float)
    scaler = StandardScaler()
    Xb_scaled = scaler.fit_transform(Xb)
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    B = pca.fit_transform(Xb_scaled)
    return scaler, pca, B


def _project_subset(X, scaler, pca):
    Xs = X.to_numpy(dtype=float)
    Xs_scaled = scaler.transform(Xs)
    Z = pca.transform(Xs_scaled)
    return Z


def _robust_limits(*arrays, low=0.5, high=99.5, pad=0.06):
    x = np.concatenate([a[:, 0] for a in arrays if len(a) > 0])
    y = np.concatenate([a[:, 1] for a in arrays if len(a) > 0])
    x0, x1 = np.percentile(x, [low, high])
    y0, y1 = np.percentile(y, [low, high])
    dx = max(1e-9, x1 - x0)
    dy = max(1e-9, y1 - y0)
    return ((x0 - pad * dx, x1 + pad * dx), (y0 - pad * dy, y1 + pad * dy))


def _draw_panel(ax, B, S, title, selected_label, selected_color, xlim, ylim, base_s, selected_s, metric_text=None):
    ax.set_facecolor("white")

    ax.scatter(
        B[:, 0], B[:, 1],
        s=base_s,
        alpha=0.78,
        color="#bbbbbb",
        edgecolors="white",
        linewidths=0.35,
        label="Original items",
        zorder=1
    )

    ax.scatter(
        S[:, 0], S[:, 1],
        s=selected_s,
        alpha=0.92,
        color=selected_color,
        edgecolors="white",
        linewidths=0.35,
        label=selected_label,
        zorder=3
    )

    if metric_text is not None:
        ax.text(
            0.03, 0.95, metric_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=18,
            color="black"
        )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=13, color="black")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.legend(frameon=True, loc="upper right", fontsize=9)


def _node_file_for_ratio(mode, ratio):
    ratio_dir = OUT_DIR / mode / str(ratio_to_int(ratio))
    return ratio_dir / f"{mode}_node0.csv"


modes_to_plot = [m for m in SWEEP_MODES if m in {"random", "padme", "graph_cut", "facility_location", "k_center"}]
target_ratios = [0.05]
panel_metric_text = {}

df_base, X_base = _load_geom_matrix(baseline_path)
scaler, pca, B = _fit_pca_reference(X_base)

proj_by_mode = {mode: {} for mode in modes_to_plot}
all_arrays = [B]

for ratio in target_ratios:
    for mode in modes_to_plot:
        if ratio not in [0.10]:
            continue

        mode_path = _node_file_for_ratio(mode, ratio)
        if not mode_path.exists():
            continue

        _, X_mode = _load_geom_matrix(mode_path)
        Z = _project_subset(X_mode, scaler, pca)
        proj_by_mode[mode][ratio] = Z
        all_arrays.append(Z)

xlim, ylim = _robust_limits(*all_arrays, low=0.5, high=99.5, pad=0.06)

selected_s = 18
base_s = selected_s * 1.1

for ratio in target_ratios:
    pct = int(round(ratio * 100))

    for mode in modes_to_plot:
        if ratio not in proj_by_mode[mode]:
            continue

        fig = plt.figure(figsize=(7, 6))
        ax = plt.gca()

        mode_label = MODE_LABELS.get(mode, mode.replace("_", " ").title())
        selected_color = MODE_STYLE.get(mode, {}).get("color", "#333333")
        metric_text = panel_metric_text.get((mode, ratio))

        _draw_panel(
            ax,
            B,
            proj_by_mode[mode][ratio],
            fr"{mode_label} ($\rho = {pct}\%$)",
            "Selected items",
            selected_color,
            xlim,
            ylim,
            base_s,
            selected_s,
            metric_text=metric_text
        )

        plt.tight_layout()
        plt.show()

In [ ]:
models = {
        "RandomForest": RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    )
}


In [ ]:
def eval_metrics(model, X_train, y_train, X_test, y_test):
    train_classes = np.unique(y_train)
    train_class_count = int(len(train_classes))

    if train_class_count < 2:
        return np.nan, np.nan, False, train_class_count

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    macro_f1 = f1_score(y_test, pred, average="macro")
    bal_acc = balanced_accuracy_score(y_test, pred)

    return macro_f1, bal_acc, True, train_class_count

rows = []

for model_name, model in models.items():

    Xb, yb, pb = load_node_dataset(BASELINE_MODE, None, BASELINE_NODE)
    base_macro_f1, base_bal_acc, base_trained, base_class_count = eval_metrics(
        model, Xb, yb, X_test, y_test
    )

    rows.append({
        "model": model_name,
        "mode": BASELINE_MODE,
        "keep_ratio": 1.0,
        "ratio_int": 100,
        "node": BASELINE_NODE,
        "trained": base_trained,
        "train_class_count": base_class_count,
        "macro_f1": base_macro_f1,
        "balanced_accuracy": base_bal_acc,
        "macro_f1_mean": base_macro_f1,
        "macro_f1_std": 0.0,
        "balanced_accuracy_mean": base_bal_acc,
        "balanced_accuracy_std": 0.0,
        "valid_nodes": 1 if base_trained else 0,
        "invalid_nodes": 0 if base_trained else 1,
        "file": pb
    })

    for mode in SWEEP_MODES:
        ratios = [0.01, 0.02, 0.03, 0.04, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60]

        for r in ratios:
            f1_scores = []
            bal_acc_scores = []
            trained_count = 0

            for i in range(NODES):
                Xn, yn, pn = load_node_dataset(mode, r, i)
                macro_f1, bal_acc, trained, train_class_count = eval_metrics(model, Xn, yn, X_test, y_test)

                if trained:
                    f1_scores.append(macro_f1)
                    bal_acc_scores.append(bal_acc)
                    trained_count += 1

                rows.append({
                    "model": model_name,
                    "mode": mode,
                    "keep_ratio": r,
                    "ratio_int": ratio_to_int(r),
                    "node": i,
                    "trained": trained,
                    "train_class_count": train_class_count,
                    "macro_f1": macro_f1,
                    "balanced_accuracy": bal_acc,
                    "file": pn
                })

            rows.append({
                "model": model_name,
                "mode": mode,
                "keep_ratio": r,
                "ratio_int": ratio_to_int(r),
                "node": "mean",
                "trained": trained_count > 0,
                "train_class_count": np.nan,
                "macro_f1_mean": np.mean(f1_scores) if f1_scores else np.nan,
                "macro_f1_std": np.std(f1_scores) if f1_scores else np.nan,
                "balanced_accuracy_mean": np.mean(bal_acc_scores) if bal_acc_scores else np.nan,
                "balanced_accuracy_std": np.std(bal_acc_scores) if bal_acc_scores else np.nan,
                "valid_nodes": trained_count,
                "invalid_nodes": NODES - trained_count
            })

results = pd.DataFrame(rows)
results[(results["node"] == "mean") | (results["mode"] == BASELINE_MODE)].sort_values(
    ["model", "mode", "keep_ratio", "node"]
)

In [ ]:
for model_name in models.keys():
    sub_mean = results[(results["model"] == model_name) & (results["node"] == "mean")].copy()
    base_row = results[(results["model"] == model_name) & (results["mode"] == BASELINE_MODE)].iloc[0]

    metrics = [
        ("macro_f1", "Macro F1"),
    ]

    for metric_key, metric_label in metrics:
        base = base_row[f"{metric_key}_mean"]

        fig = plt.figure(figsize=(7, 6))
        ax = plt.gca()

        ax.axhline(base, linestyle="--", linewidth=2, color="black", label=MODE_LABELS[BASELINE_MODE])

        for mode in SWEEP_MODES:
            s = sub_mean[sub_mean["mode"] == mode].sort_values("keep_ratio")
            if s.empty:
                continue
            ax.errorbar(
                s["keep_ratio"],
                s[f"{metric_key}_mean"],
                marker=MODE_STYLE[mode]["marker"],
                color=MODE_STYLE[mode]["color"],
                markersize=7,
                linewidth=2,
                capsize=4,
                label=MODE_LABELS[mode]
            )

        ax.set_xlabel(r"Data keep ratio ($\rho$)", fontsize=13)
        ax.set_ylabel(f"{metric_label}", fontsize=13)
        ax.tick_params(axis="both", labelsize=11)
        ax.legend(fontsize=11)

        plt.tight_layout()
        plt.show()